In [31]:
from transformers import AutoModel, AutoTokenizer, AutoConfig, default_data_collator
from torch.utils.data import DataLoader
from datasets import load_dataset
from tqdm.notebook import tqdm
import pandas as pd
import torch
import os

In [27]:
device = 'cuda'

In [26]:
def save_tensor_chunk(tensor, out_dir, chunk_idx):
    print(out_dir)
    for layer_idx in range(tensor.shape[0]):
        layer_dir = os.path.join(out_dir, str(layer_idx))
        if not os.path.exists(layer_dir):
            os.makedirs(layer_dir)
        out_path = os.path.join(layer_dir, f'{chunk_idx}.pt')
        torch.save(tensor[layer_idx], out_path)



In [47]:
sentences_df
tokenizer_path = '/home/luca/Workspace/curriculum_learning/models/bert_tokenizer'
model_path = '/home/luca/Workspace/curriculum_learning/models/bert_base_42_train_1_orig'
out_dir = '/home/luca/Workspace/curriculum_learning/data/representations/prova'

In [36]:
batch_size = 16

In [35]:
model = AutoModel.from_pretrained(model_path, output_hidden_states=True)
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path, return_tensors='pt') # use_fast=False,
model.to(device);

Some weights of BertModel were not initialized from the model checkpoint at /home/luca/Workspace/curriculum_learning/models/bert_base_42_train_1_orig and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [48]:
def preprocess_dataset(dataset, tokenizer):
    def preprocessing_function(examples):
        result = tokenizer(examples['text'], padding='max_length', max_length=512, truncation=True)
        return result

    # cols_to_remove = dataset.column_names
    # cols_to_remove.remove('text')
    tokenized_dataset = dataset.map(
        preprocessing_function,
        batched=True,
        # remove_columns=cols_to_remove,
        desc="Running tokenizer on dataset",
    )
    return tokenized_dataset

In [49]:
data_files = {'train': sentences_path}

dataset = load_dataset('csv', data_files=data_files, sep='\t', column_names=['sentence_id', 'text'])
dataset = preprocess_dataset(dataset['train'], tokenizer)

dataloader = DataLoader(dataset, shuffle=False, collate_fn=default_data_collator,
                                batch_size=batch_size)

In [50]:
all_hidden_states = None
model.eval()
with torch.no_grad():
    for batch_cpu in tqdm(dataloader):
        batch = {key: value.to(device) for key, value in batch_cpu.items()}
        hidden_states = model(**batch)['hidden_states']        
        non_pad_tokens = batch['attention_mask'].sum(axis=1)
        hidden_states = torch.stack(hidden_states, dim=1)

        mask = batch['attention_mask']
        mask = mask.view(mask.shape[0], 1, mask.shape[1], 1).expand(hidden_states.shape)
        masked_hidden_states = hidden_states * mask
        masked_hidden_states = masked_hidden_states[:, 1:, :, :]
        sum_hidden_states = torch.sum(masked_hidden_states, dim=2)
        average_hidden_states = torch.div(sum_hidden_states,
                                            non_pad_tokens.view(-1, 1, 1))  # batch_size, num_layers + 1, hidden_size
        average_hidden_states = average_hidden_states.transpose(0, 1)
        all_hidden_states = average_hidden_states if all_hidden_states is None else torch.cat(
            (all_hidden_states, average_hidden_states), dim=1)
        
    
    save_tensor(all_hidden_states, out_dir)

  0%|          | 0/2149 [00:00<?, ?it/s]

/home/luca/Workspace/curriculum_learning/data/representations/prova
